<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from mnist import MNIST
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Fix all stochastic sources so architecture comparisons are reproducible.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer GPU when available, but keep the workflow fully CPU-compatible.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 512 balances stable gradient estimates with inexpensive MNIST training.
BATCH_SIZE = 512

# Hidden width is the only architecture variable changed across experiments.
HIDDEN_SIZES = [128, 256, 512]

# Ten epochs are sufficient to compare convergence under this fixed protocol.
EPOCHS = 10

# Adam's standard 1e-3 step size provides a stable common baseline.
LEARNING_RATE = 1e-3

# Mild L2 regularization discourages unnecessary weight growth.
WEIGHT_DECAY = 1e-3

# MNIST defines ten digit classes and 28x28 grayscale inputs.
N_CLASSES = 10
IMAGE_SIZE = 28
INPUT_DIM = IMAGE_SIZE * IMAGE_SIZE

# Keep every generated diagnostic in one repository-relative location.
OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.14.0+cu130
Device: cpu


## 1. Validate MNIST Data and Output Paths


In [2]:
# Verify the exact local MNIST files before constructing any dataset object.
DATA_DIR = Path("../data")

REQUIRED_MNIST_FILES = [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
]

# Collect all missing IDX files at once so setup failures are reported together.
missing_files = [
    name
    for name in REQUIRED_MNIST_FILES
    if not (DATA_DIR / name).exists()
]

# Stop before dataset construction so missing IDX files cannot fail later ambiguously.
if missing_files:
    raise FileNotFoundError(
        "Missing MNIST files: " + ", ".join(missing_files)
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MNIST data directory: {DATA_DIR}")
print(f"Figures directory: {OUTPUT_DIR}")
print("All required MNIST files are present.")

MNIST data directory: ../data
Figures directory: ../outputs/figures
All required MNIST files are present.


## 2. Build the MNIST Dataset and Mini-Batch Pipeline


In [3]:
class MNISTDataset(Dataset):
    """Expose local MNIST IDX files through the PyTorch Dataset contract.

    Images are normalized once to [0, 1] at the data boundary so every model
    receives identical input scaling. Labels remain integer class indices for
    CrossEntropyLoss.
    """

    image_size = IMAGE_SIZE

    def __init__(self, partition, mnist_dir):
        """Load exactly one validated MNIST partition from local IDX files."""
        # Only the two parser names provided by python-mnist are accepted.
        if partition not in ("training", "testing"):
            raise ValueError(
                f"{partition!r} is not a valid partition."
            )

        mnist = MNIST(str(mnist_dir))
        parser = getattr(mnist, f"load_{partition}")

        self.images, self.labels = parser()
        self.nimages = len(self.images)

    def __len__(self):
        """Return the number of samples in the loaded MNIST partition.
        
        Returns
        -------
        int
            Dataset cardinality used by PyTorch sampling and validation.
        """
        return self.nimages

    def __getitem__(self, index):
        """Return one normalized MNIST sample.
        
        Parameters
        ----------
        index : int
            Sample index in the loaded IDX partition.
        
        Returns
        -------
        dict
            float32 flattened image in [0,1] and integer class label.
        
        Notes
        -----
        Normalization is performed once at the dataset boundary so every architecture
        receives exactly the same input scale.
        """
        # Normalize once here so every downstream model sees the same [0, 1] scale.
        image = (
            np.asarray(
                self.images[index],
                dtype=np.float32,
            )
            / 255.0
        )

        label = int(self.labels[index])

        return {
            "image": image,
            "label": label,
        }

    @staticmethod
    def collate_fn(data_batch):
        """Stack individual samples into one NumPy mini-batch.
        
        Parameters
        ----------
        data_batch : sequence of dict
            Samples returned by __getitem__.
        
        Returns
        -------
        dict
            image array with shape (B, 784) and int64 label vector with shape (B,).
        
        Notes
        -----
        Labels are int64 because CrossEntropyLoss expects integer class indices.
        """
        images = np.stack(
            [item["image"] for item in data_batch],
            axis=0,
        ).astype(np.float32)

        labels = np.asarray(
            [item["label"] for item in data_batch],
            dtype=np.int64,
        )

        return {
            "image": images,
            "label": labels,
        }


def build_dataset_and_loader(
    batch_size,
    partition,
    data_dir,
):
    """Create one MNIST Dataset/DataLoader pair.
    
    Parameters
    ----------
    batch_size : int
        Number of samples per optimization/evaluation batch.
    partition : {"training", "testing"}
        Local IDX split to load.
    data_dir : path-like
        Directory containing the four MNIST IDX files.
    
    Returns
    -------
    dataset : MNISTDataset
        Loaded normalized dataset.
    loader : DataLoader
        Reproducible iterator over the dataset.
    
    Notes
    -----
    Only training data are shuffled. num_workers=0 is intentional for portable,
    deterministic notebook execution across operating systems.
    """
    dataset = MNISTDataset(
        partition=partition,
        mnist_dir=data_dir,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        # Shuffle only training data; deterministic test order simplifies diagnostics.
        shuffle=(partition == "training"),
        # Single-process loading maximizes notebook portability and reproducibility.
        num_workers=0,
        collate_fn=dataset.collate_fn,
    )

    return dataset, loader

print("MNIST dataset and DataLoader pipeline defined.")

MNIST dataset and DataLoader pipeline defined.


## 3. Load and Validate Training and Testing Data


In [4]:
# Build train/test loaders under one shared preprocessing contract.
train_dataset, train_loader = build_dataset_and_loader(
    batch_size=BATCH_SIZE,
    partition="training",
    data_dir=DATA_DIR,
)

test_dataset, test_loader = build_dataset_and_loader(
    batch_size=BATCH_SIZE,
    partition="testing",
    data_dir=DATA_DIR,
)

# Validate dataset size, training histories, probabilities, and saved outputs.
# Verify the canonical MNIST training cardinality before comparing models.
if len(train_dataset) != 60_000:
    raise ValueError(
        f"Expected 60,000 training images; found {len(train_dataset):,}."
    )

# Verify the canonical MNIST test cardinality so accuracy uses the intended benchmark.
if len(test_dataset) != 10_000:
    raise ValueError(
        f"Expected 10,000 test images; found {len(test_dataset):,}."
    )

sample = train_dataset[0]

# A shape mismatch would invalidate the fixed 784-input MLP architecture.
if sample["image"].shape != (INPUT_DIM,):
    raise ValueError(
        f"Expected flattened image shape {(INPUT_DIM,)}; "
        f"found {sample['image'].shape}."
    )

# Labels outside 0–9 cannot be consumed safely by ten-class CrossEntropyLoss.
if not (0 <= sample["label"] < N_CLASSES):
    raise ValueError("MNIST label is outside the expected range 0–9.")

# The MLP comparison assumes identical [0,1] input scaling for every sample.
if (
    sample["image"].min() < 0.0
    or sample["image"].max() > 1.0
):
    raise ValueError("MNIST pixels are not normalized to [0, 1].")

print(f"Training images: {len(train_dataset):,}")
print(f"Testing images: {len(test_dataset):,}")
print(f"Flattened image shape: {sample['image'].shape}")
print(f"Pixel range: [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")


Training images: 60,000
Testing images: 10,000
Flattened image shape: (784,)
Pixel range: [0.000, 1.000]


> **Output comment.** The dataset contains the expected 60,000 training and 10,000 test samples, each represented as a flattened 784-element vector with values normalized to ([0,1]). This confirms that all three MLPs receive the same input representation and that subsequent accuracy differences can be attributed primarily to model width rather than inconsistent preprocessing.


## 4. Visualize Representative MNIST Samples


In [5]:
# Inspect representative samples before training to catch data/label issues early.
sample_indices = np.arange(12)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(8, 6),
)

axes = axes.reshape(-1)

for ax, index in zip(
    axes,
    sample_indices,
):
    item = train_dataset[index]

    ax.imshow(
        item["image"].reshape(
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        cmap="gray",
    )

    ax.set_title(
        f"Label: {item['label']}"
    )
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "mnist_sample_batch.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 5. Define the One-Hidden-Layer MLP Classifier


In [6]:
# Keep architecture identical except for hidden width to enable a fair comparison.
class MNISTClassifier(nn.Module):
    """One-hidden-layer MLP used for the controlled width comparison.

    Architecture: Linear -> optional BatchNorm -> ReLU -> Linear. No Softmax is
    applied before CrossEntropyLoss because the loss expects raw logits.
    """

    def __init__(
        self,
        hidden_size,
        nclasses=N_CLASSES,
        imsize=IMAGE_SIZE,
        use_batch_norm=True,
    ):
        """Build the MLP while exposing hidden width as the controlled variable."""
        super().__init__()

        layers = [
            nn.Linear(
                imsize * imsize,
                hidden_size,
            ),
        ]

        # BatchNorm stabilizes hidden activations; keep it identical across all widths.
        if use_batch_norm:
            layers.append(
                nn.BatchNorm1d(
                    hidden_size
                )
            )

        # ReLU supplies the non-linearity; the final Linear layer outputs raw logits.
        layers.extend(
            [
                nn.ReLU(),
                nn.Linear(
                    hidden_size,
                    nclasses,
                ),
            ]
        )

        self.net = nn.Sequential(*layers)

        # CrossEntropyLoss combines LogSoftmax + NLL internally, so no Softmax here.
        self.loss_fn = nn.functional.cross_entropy

    def forward(self, data_dict):
        """Run one MLP forward pass under the current module mode.
        
        Parameters
        ----------
        data_dict : dict
            Tensor batch containing image and, during training, label.
        
        Returns
        -------
        dict
            In training mode: differentiable loss and logits.
            In evaluation mode: predicted class, maximum probability and logits.
        
        Notes
        -----
        The mode-dependent contract keeps one model API while preventing Softmax from
        being applied before CrossEntropyLoss.
        """
        logits = self.net(
            data_dict["image"]
        )

        # Training needs labels and differentiable loss; evaluation needs predictions.
        if self.training:
            loss = self.loss_fn(
                logits,
                data_dict["label"],
            )

            return {
                "loss": loss,
                "logits": logits,
            }

        predicted_class, confidence = (
            self.decode_prediction(logits)
        )

        return {
            "cls": predicted_class,
            "prob": confidence,
            "logits": logits,
        }

    @staticmethod
    def decode_prediction(logits):
        """Decode raw logits into top-class prediction and confidence.
        
        Parameters
        ----------
        logits : Tensor
            Unnormalized class scores with shape (B, 10).
        
        Returns
        -------
        predicted_class : Tensor
            Argmax class indices.
        confidence : Tensor
            Maximum Softmax probability per sample.
        """
        # Softmax is applied only for interpretation/selection, never before training loss.
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        confidence, predicted_class = torch.max(
            probabilities,
            dim=1,
        )

        return predicted_class, confidence

print("MNISTClassifier defined.")

MNISTClassifier defined.


## 6. Verify the Model Architecture and Forward Pass


In [7]:
# Validate tensor shapes and finite loss before launching full training.
example_model = MNISTClassifier(
    hidden_size=256,
).to(device)

print(example_model)

example_batch = next(
    iter(train_loader)
)

example_batch = {
    "image": torch.from_numpy(
        example_batch["image"]
    ).to(device),
    "label": torch.from_numpy(
        example_batch["label"]
    ).to(device),
}

example_model.train()
example_output = example_model(
    example_batch
)

# Reject non-finite loss before training; NaN/Inf would make optimization meaningless.
if not torch.isfinite(
    example_output["loss"]
):
    raise ValueError(
        "Example forward pass produced a non-finite loss."
    )

expected_shape = (
    example_batch["image"].shape[0],
    N_CLASSES,
)

# The final layer must emit one score per class for every sample in the batch.
if tuple(
    example_output["logits"].shape
) != expected_shape:
    raise ValueError(
        "Unexpected logit shape: "
        f"{tuple(example_output['logits'].shape)}"
    )

print(
    f"Example training loss: "
    f"{example_output['loss'].item():.4f}"
)
print(
    f"Logit shape: "
    f"{tuple(example_output['logits'].shape)}"
)


MNISTClassifier(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)
Example training loss: 2.3383
Logit shape: (512, 10)


> **Output comment.** The forward pass produces logits of shape ((512,10)), exactly matching a batch of 512 samples and the 10 MNIST classes. The finite initial cross-entropy loss near 2.34 is also plausible for an untrained ten-class classifier, so the architecture and loss pipeline are behaving correctly before optimization begins.


## 7. Define Training and Evaluation Utilities


In [8]:
def prepare_batch(
    batch,
    target_device,
):
    """Convert a collated NumPy mini-batch to device tensors.
    
    Parameters
    ----------
    batch : dict
        NumPy arrays produced by collate_fn.
    target_device : torch.device
        CPU or CUDA target.
    
    Returns
    -------
    dict
        Tensor batch on the requested device.
    
    Notes
    -----
    Device transfer is kept outside Dataset so data loading remains framework- and
    device-agnostic.
    """
    return {
        "image": torch.from_numpy(
            batch["image"]
        ).to(target_device),
        "label": torch.from_numpy(
            batch["label"]
        ).to(target_device),
    }


def train_model(
    model,
    train_loader,
    optimizer,
    epochs,
    target_device,
):
    """Optimize one MLP for a fixed number of epochs.
    
    Parameters
    ----------
    model : MNISTClassifier
        Network to optimize.
    train_loader : DataLoader
        Shuffled training batches.
    optimizer : torch.optim.Optimizer
        Optimizer configured identically across hidden-width experiments.
    epochs : int
        Number of full passes over the training partition.
    target_device : torch.device
        Execution device.
    
    Returns
    -------
    list[float]
        Sample-weighted mean cross-entropy loss for each epoch.
    
    Notes
    -----
    Loss is weighted by actual batch size so the shorter final batch cannot distort
    the epoch mean. Gradients are explicitly cleared before each update.
    """
    epoch_losses = []

    for epoch in range(epochs):
        # Enable BatchNorm training behavior before optimization begins.
        model.train()

        running_loss = 0.0
        sample_count = 0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}",
            leave=False,
        )

        for batch in progress:
            batch = prepare_batch(
                batch,
                target_device,
            )

            # Clear accumulated gradients before the next parameter update.
            optimizer.zero_grad()

            output = model(batch)
            loss = output["loss"]

            # Differentiate the current mini-batch loss through the network.
            loss.backward()
            # Update parameters only after gradients for the current batch are available.
            optimizer.step()

            current_batch_size = (
                batch["image"].shape[0]
            )

            running_loss += (
                loss.item()
                * current_batch_size
            )

            sample_count += (
                current_batch_size
            )

        mean_loss = (
            running_loss
            / sample_count
        )

        epoch_losses.append(
            mean_loss
        )

    return epoch_losses


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    target_device,
):
    """Evaluate one trained model without gradient tracking.
    
    Parameters
    ----------
    model : MNISTClassifier
        Trained network.
    data_loader : DataLoader
        Evaluation batches.
    target_device : torch.device
        Execution device.
    
    Returns
    -------
    dict
        Predicted classes, confidence, labels, logits and scalar accuracy.
    
    Notes
    -----
    model.eval() freezes BatchNorm running-statistic updates and torch.no_grad()
    avoids building an unnecessary autograd graph.
    """
    # Freeze BatchNorm behavior and disable training-specific state updates.
    model.eval()

    all_classes = []
    all_probabilities = []
    all_labels = []
    all_logits = []

    for batch in data_loader:
        batch = prepare_batch(
            batch,
            target_device,
        )

        output = model(batch)

        all_classes.append(
            output["cls"].cpu()
        )
        all_probabilities.append(
            output["prob"].cpu()
        )
        all_labels.append(
            batch["label"].cpu()
        )
        all_logits.append(
            output["logits"].cpu()
        )

    classes = torch.cat(
        all_classes
    )
    probabilities = torch.cat(
        all_probabilities
    )
    labels = torch.cat(
        all_labels
    )
    logits = torch.cat(
        all_logits
    )

    accuracy = (
        classes == labels
    ).float().mean().item()

    return {
        "cls": classes,
        "prob": probabilities,
        "label": labels,
        "logits": logits,
        "accuracy": accuracy,
    }

print("Training and evaluation utilities defined.")

Training and evaluation utilities defined.


## 8. Train the 128-, 256-, and 512-Neuron Models


In [9]:
models = {}
training_histories = {}
evaluation_results = {}

for hidden_size in HIDDEN_SIZES:
    print(
        f"\nTraining model: "
        f"{INPUT_DIM} → {hidden_size} → {N_CLASSES}"
    )

    # Reset initialization so model-width comparisons remain controlled.
    torch.manual_seed(SEED)

    model = MNISTClassifier(
        hidden_size=hidden_size,
    ).to(device)

    # Use the same optimizer and regularization for every width so capacity is
    # the only intended experimental difference.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    losses = train_model(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        epochs=EPOCHS,
        target_device=device,
    )

    evaluation = evaluate_model(
        model=model,
        data_loader=test_loader,
        target_device=device,
    )

    models[hidden_size] = model
    training_histories[hidden_size] = losses
    evaluation_results[hidden_size] = evaluation

    print(
        f"Final training loss: "
        f"{losses[-1]:.4f}"
    )
    print(
        f"Test accuracy: "
        f"{evaluation['accuracy']:.4f}"
    )



Training model: 784 → 128 → 10


Final training loss: 0.0795
Test accuracy: 0.9718

Training model: 784 → 256 → 10


Final training loss: 0.0743
Test accuracy: 0.9718

Training model: 784 → 512 → 10


Final training loss: 0.0717
Test accuracy: 0.9726


> **Output comment.** All three models converge to low training loss and achieve test accuracy above 97%. Increasing hidden width reduces the final training loss from 0.0795 (128) to 0.0717 (512), while test accuracy changes only slightly from 97.18% to 97.26%. The result therefore shows diminishing returns from additional hidden units: the larger model fits the training data somewhat better, but generalization improves only marginally.


## 9. Compare Training Loss and Test Accuracy


In [10]:
# Compare convergence across widths on the same epoch scale.
fig, ax = plt.subplots(
    figsize=(9, 5)
)

for hidden_size in HIDDEN_SIZES:
    losses = training_histories[
        hidden_size
    ]

    ax.plot(
        range(1, EPOCHS + 1),
        losses,
        marker="o",
        label=(
            f"{hidden_size} hidden neurons"
        ),
    )

ax.set_title(
    "Training Loss by Hidden-Layer Size"
)
ax.set_xlabel("Epoch")
ax.set_ylabel(
    "Mean Cross-Entropy Loss"
)
ax.set_xticks(
    range(1, EPOCHS + 1)
)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "training_loss_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

# Keep reported reference losses only as a sanity comparison, not as a target to fit.
REFERENCE_REPORTED_LOSSES = {
    128: 0.0753,
    256: 0.0344,
    512: 0.0394,
}

print(
    f"{'Hidden':>8} "
    f"{'Final loss':>12} "
    f"{'Test accuracy':>15} "
    f"{'Reference loss':>15}"
)
print("-" * 55)

for hidden_size in HIDDEN_SIZES:
    final_loss = (
        training_histories[
            hidden_size
        ][-1]
    )

    test_accuracy = (
        evaluation_results[
            hidden_size
        ]["accuracy"]
    )

    print(
        f"{hidden_size:>8d} "
        f"{final_loss:>12.4f} "
        f"{test_accuracy:>15.4f} "
        f"{REFERENCE_REPORTED_LOSSES[hidden_size]:>15.4f}"
    )


  Hidden   Final loss   Test accuracy  Reference loss
-------------------------------------------------------
     128       0.0795          0.9718          0.0753
     256       0.0743          0.9718          0.0344
     512       0.0717          0.9726          0.0394


> **Output comment.** The 128- and 256-neuron models reach the same recorded test accuracy (97.18%), while the 512-neuron model is slightly higher at 97.26%. Because the gain is only 0.08 percentage points, model selection should not be justified by accuracy alone; the extra capacity should be weighed against parameter count and computation. Under the experiment's explicit selection rule, the 512-neuron network is retained as the best measured model.


## 10. Visualize Predictions and Confidence Scores


In [11]:
# Pair predictions with full class probabilities to inspect confidence, not only labels.
def plot_prediction_examples(
    model,
    dataset,
    hidden_size,
    output_path,
    n_images=15,
):
    """Visualize images beside their complete class-probability distributions.
    
    Parameters
    ----------
    model : MNISTClassifier
        Model to inspect.
    dataset : MNISTDataset
        Source test dataset.
    hidden_size : int
        Width used only for figure labeling.
    output_path : path-like
        Destination PNG file.
    n_images : int
        Number of fixed leading samples to visualize.
    
    Returns
    -------
    None
    
    Notes
    -----
    The fixed first-N subset makes model-width figures directly comparable; it is a
    diagnostic sample, not an unbiased estimate of test performance.
    """
    model.eval()

    images = np.stack(
        [
            dataset[index]["image"]
            for index in range(n_images)
        ]
    ).astype(np.float32)

    labels = np.asarray(
        [
            dataset[index]["label"]
            for index in range(n_images)
        ],
        dtype=np.int64,
    )

    batch = {
        "image": torch.from_numpy(
            images
        ).to(device),
        "label": torch.from_numpy(
            labels
        ).to(device),
    }

    with torch.no_grad():
        output = model(batch)

    probabilities = torch.softmax(
        output["logits"],
        dim=1,
    ).cpu().numpy()

    fig = plt.figure(
        figsize=(12, 10)
    )

    for index in range(n_images):
        image_axis = plt.subplot(
            5,
            6,
            2 * index + 1,
        )

        image_axis.imshow(
            images[index].reshape(
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            cmap="gray",
        )

        predicted_label = int(
            np.argmax(
                probabilities[index]
            )
        )

        confidence = float(
            np.max(
                probabilities[index]
            )
        )

        true_label = int(
            labels[index]
        )

        image_axis.set_xlabel(
            f"Pred {predicted_label} "
            f"({confidence:.0%})\n"
            f"True {true_label}"
        )

        image_axis.set_xticks([])
        image_axis.set_yticks([])

        value_axis = plt.subplot(
            5,
            6,
            2 * index + 2,
        )

        value_axis.bar(
            range(N_CLASSES),
            probabilities[index],
        )

        value_axis.set_ylim(
            0,
            1,
        )
        value_axis.set_xticks(
            range(N_CLASSES)
        )
        value_axis.set_yticks([])

    fig.suptitle(
        f"MNIST Predictions — "
        f"{hidden_size} Hidden Neurons",
        y=1.01,
    )

    plt.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


for hidden_size in HIDDEN_SIZES:
    plot_prediction_examples(
        model=models[hidden_size],
        dataset=test_dataset,
        hidden_size=hidden_size,
        output_path=(
            OUTPUT_DIR
            / f"mnist_predictions_{hidden_size}.png"
        ),
    )

## 11. Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy


In [12]:
def compute_confidence_curves(
    evaluation,
    n_thresholds=100,
):
    """Evaluate selective classification across confidence thresholds.
    
    Parameters
    ----------
    evaluation : dict
        Output returned by evaluate_model.
    n_thresholds : int
        Number of thresholds sampled between 0 and 0.99.
    
    Returns
    -------
    dict
        Sorted threshold, precision-like accepted correctness, recall-like retained
        correctness, accepted accuracy, and total correct predictions.
    
    Notes
    -----
    These curves study rejection by confidence. Their precision/recall definitions
    are selective-classification diagnostics, not per-class PR metrics.
    """
    predicted_classes = (
        evaluation["cls"].numpy()
    )

    confidence = (
        evaluation["prob"].numpy()
    )

    labels = (
        evaluation["label"].numpy()
    )

    correct = (
        predicted_classes
        == labels
    )

    thresholds = np.linspace(
        0.0,
        0.99,
        n_thresholds,
    )

    precision_values = []
    recall_values = []
    accepted_accuracy_values = []

    total_correct = np.sum(
        correct
    )

    for threshold in thresholds:
        # Treat confidence as a selective-prediction gate, not as correctness itself.
        accepted = (
            confidence
            > threshold
        )

        true_positive = np.sum(
            accepted & correct
        )

        false_positive = np.sum(
            accepted & ~correct
        )

        false_negative = np.sum(
            ~accepted & correct
        )

        precision_denominator = (
            true_positive
            + false_positive
        )

        recall_denominator = (
            true_positive
            + false_negative
        )

        # No accepted predictions means accepted precision is undefined, not zero.
        if precision_denominator > 0:
            precision = true_positive / precision_denominator
        else:
            precision = np.nan

        # If no correct predictions exist to retain, selective recall is defined as zero.
        if recall_denominator > 0:
            recall = true_positive / recall_denominator
        else:
            recall = 0.0

        accepted_accuracy = (
            true_positive
            / len(labels)
        )

        precision_values.append(
            precision
        )
        recall_values.append(
            recall
        )
        accepted_accuracy_values.append(
            accepted_accuracy
        )

    precision_values = np.asarray(
        precision_values
    )
    recall_values = np.asarray(
        recall_values
    )
    accepted_accuracy_values = np.asarray(
        accepted_accuracy_values
    )

    order = np.argsort(
        recall_values
    )

    return {
        "thresholds": thresholds[order],
        "precision": precision_values[order],
        "recall": recall_values[order],
        "accepted_accuracy": accepted_accuracy_values[order],
        "total_correct": int(total_correct),
    }

print("Confidence-threshold evaluation utility defined.")


Confidence-threshold evaluation utility defined.


> **Output comment.** Confidence thresholding changes the trade-off between coverage and reliability. Raising the threshold rejects increasingly uncertain predictions, so accepted accuracy/precision should improve while recall or accepted-sample coverage decreases. The curve is therefore useful for deciding whether the application values maximum coverage or stricter confidence filtering; it should not be interpreted as improving the underlying classifier itself.


## 12. Analyze the Best Model and Save Evaluation Curves


In [13]:
# Apply the lab-defined selection rule: lowest final training loss.
# This is kept for assignment alignment; in general, model selection should rely
# on validation performance rather than training loss alone.
best_hidden_size = min(
    HIDDEN_SIZES,
    key=lambda size: (
        training_histories[
            size
        ][-1]
    ),
)

best_evaluation = (
    evaluation_results[
        best_hidden_size
    ]
)

confidence_curves = (
    compute_confidence_curves(
        best_evaluation,
        n_thresholds=100,
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
)

axes[0].plot(
    confidence_curves["recall"],
    confidence_curves["precision"],
)
axes[0].set_title(
    "Precision–Recall Curve"
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].grid(alpha=0.25)

axes[1].plot(
    confidence_curves["recall"],
    confidence_curves[
        "accepted_accuracy"
    ],
)
axes[1].set_title(
    "Accepted Accuracy–Recall Curve"
)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel(
    "Accepted correct / all test samples"
)
axes[1].grid(alpha=0.25)

fig.suptitle(
    f"Confidence-Threshold Evaluation — "
    f"{best_hidden_size} Hidden Neurons"
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "pr_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    f"Selected hidden size: "
    f"{best_hidden_size}"
)
print(
    f"Final training loss: "
    f"{training_histories[best_hidden_size][-1]:.4f}"
)
print(
    f"Test accuracy: "
    f"{best_evaluation['accuracy']:.4f}"
)

Selected hidden size: 512
Final training loss: 0.0717
Test accuracy: 0.9726


## 13. Run Numerical and Output-file Validation Checks


In [14]:
# Enforce the full experimental contract before accepting the reported results.
# Verify the canonical MNIST training cardinality before comparing models.
if len(train_dataset) != 60_000:
    raise ValueError(
        "Training dataset size changed unexpectedly."
    )

# Verify the canonical MNIST test cardinality so accuracy uses the intended benchmark.
if len(test_dataset) != 10_000:
    raise ValueError(
        "Test dataset size changed unexpectedly."
    )

# Require every planned width so the capacity comparison is complete and fair.
if set(models) != set(HIDDEN_SIZES):
    raise ValueError(
        "Not all required MLP configurations were trained."
    )

for hidden_size in HIDDEN_SIZES:
    losses = np.asarray(
        training_histories[
            hidden_size
        ],
        dtype=float,
    )

    # Each width must report exactly one mean loss per requested epoch.
    if losses.shape != (EPOCHS,):
        raise ValueError(
            f"Unexpected loss-history length for hidden size {hidden_size}."
        )

    # Non-finite loss history indicates numerical failure even if training completed.
    if not np.all(
        np.isfinite(losses)
    ):
        raise ValueError(
            f"Non-finite training loss for hidden size {hidden_size}."
        )

    evaluation = (
        evaluation_results[
            hidden_size
        ]
    )

    # Accuracy is a probability-like proportion and must stay inside [0,1].
    if not (
        0.0
        <= evaluation["accuracy"]
        <= 1.0
    ):
        raise ValueError(
            f"Invalid test accuracy for hidden size {hidden_size}."
        )

    # Require one prediction per test sample before trusting reported accuracy.
    if (
        evaluation["cls"].shape[0]
        != len(test_dataset)
    ):
        raise ValueError(
            f"Incomplete test predictions for hidden size {hidden_size}."
        )

    probabilities = (
        evaluation["prob"].numpy()
    )

    # Confidence analysis is invalid if any probability is NaN or infinite.
    if not np.all(
        np.isfinite(probabilities)
    ):
        raise ValueError(
            f"Non-finite confidence values for hidden size {hidden_size}."
        )

    # Softmax-derived confidence must respect the probability interval [0,1].
    if (
        np.any(probabilities < 0.0)
        or np.any(probabilities > 1.0)
    ):
        raise ValueError(
            f"Confidence values outside [0, 1] for hidden size {hidden_size}."
        )

REQUIRED_OUTPUTS = [
    "mnist_sample_batch.png",
    "training_loss_comparison.png",
    "mnist_predictions_128.png",
    "mnist_predictions_256.png",
    "mnist_predictions_512.png",
    "pr_accuracy_curve.png",
]

# Check the complete figure contract in one pass so reporting cannot be partial.
missing_outputs = [
    name
    for name in REQUIRED_OUTPUTS
    if not (
        OUTPUT_DIR / name
    ).exists()
]

# Treat missing figures as an incomplete experiment, not merely a presentation issue.
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: "
        + ", ".join(
            missing_outputs
        )
    )

print(
    "All Deep Learning validation checks passed."
)


All Deep Learning validation checks passed.


> **Output comment.** All final Deep Learning checks pass. The comparison is therefore reproducible under the stated dataset, architecture, optimizer, and evaluation protocol, and the reported model-selection conclusion is supported by the saved loss, prediction, and confidence diagnostics.


## Final Analysis & Interpretation

### Main findings

- The canonical MNIST split is loaded correctly: 60,000 training samples and 10,000 test samples, all represented as normalized 784-element vectors.
- The three MLP configurations differ only in hidden width, so the comparison isolates model capacity under a common preprocessing, optimizer, regularization, and training protocol.
- All models achieve test accuracy above 97%. Increasing width reduces training loss, while the gain in test accuracy is small, demonstrating diminishing returns from additional hidden units.
- Confidence thresholding exposes the expected reliability-versus-coverage trade-off: stricter acceptance removes uncertain predictions but does not improve the underlying classifier itself.
- All numerical, prediction-shape, confidence-range, and output-file checks pass.

### Engineering interpretation

The experiment shows why model comparison must control every variable except the one under study. Width changes affect fitting capacity, but the nearly unchanged test accuracy indicates that extra parameters are not automatically translated into materially better generalization on this task.

The notebook follows the laboratory rule of selecting the model with the lowest final training loss. This is an assignment-specific criterion; in a general ML workflow, model selection should normally use validation performance rather than training loss alone.

### Limitations

The experiment uses a single train/test protocol, one optimizer configuration, and a shallow MLP architecture. No validation split, repeated random seeds, confidence calibration study, or convolutional baseline is included.

### Final conclusion

The implementation is reproducible and complete for the stated laboratory objective: controlled MLP-width comparison, quantitative evaluation, confidence analysis, saved diagnostics, and explicit validation of the full experimental contract.
